# Test: NPlay simulator connection

Connects to a running Blackrock NPlay instance (replaying a `.nev`/`.ns5`
recording), runs the full pipeline in real-time, and reports detected events.

NPlay is an **online** source — it streams packets over UDP exactly like
the real NSP hardware, so the pipeline exercises the same live code path
as a real recording session. The difference is that NPlay replays a saved
file rather than acquiring from electrodes.

**Requirements:**
- NPlay running and replaying a file (via Blackrock Central Suite)
- `pip install direct-neural-biasing[live]`

Edit `N_CHANNELS` and `RUN_SECONDS` below to match your replay file.

In [ ]:
# === Configuration — edit for your NPlay session ===
N_CHANNELS = 83          # must match the replayed file
SAMPLE_RATE = 30000.0
RUN_SECONDS = 30         # how long to acquire (or until NPlay stops)
PROTOCOL = "NPLAY"       # pycbsdk protocol string

In [ ]:
import logging
import threading
import time

from dnb import NPlaySource, Pipeline, PipelineConfig, EventType
from dnb.modules import WaveletConvolution, EventDetector, PowerEstimator
from dnb.io import ResultWriter

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")

config = PipelineConfig(
    sample_rate=SAMPLE_RATE,
    n_channels=N_CHANNELS,
    chunk_duration=0.2,
    buffer_duration=10.0,
)

writer = ResultWriter(output_dir="./nplay_output", prefix="nplay")

pipeline = Pipeline(
    source=NPlaySource(protocol=PROTOCOL),
    modules=[
        WaveletConvolution(freq_min=1, freq_max=200, n_freqs=40),
        PowerEstimator(),
        EventDetector(
            event_type=EventType.RIPPLE,
            freq_range=(80, 250),
            threshold_std=3.0,
            warmup_chunks=10,
        ),
    ],
    config=config,
)

# Track events and per-chunk timing
event_log = []
chunk_times = []
chunk_count = [0]


def on_event(event):
    event_log.append(event)
    writer.on_event(event)
    if len(event_log) <= 20:
        print(
            f"  [{len(event_log):3d}] {event.event_type.name} "
            f"ch={event.channel_id} t={event.timestamp:.3f}s "
            f"dur={event.duration:.3f}s"
        )


pipeline.on_event(None, on_event)

print(f"Connecting to NPlay ({PROTOCOL})...")
print(f"  Channels: {N_CHANNELS}, Sample rate: {SAMPLE_RATE:.0f} Hz")
print(f"  Running for up to {RUN_SECONDS}s (or until NPlay stops)")
print()

timer = threading.Timer(RUN_SECONDS, pipeline.stop)
timer.start()

try:
    pipeline.run_online()
except KeyboardInterrupt:
    pipeline.stop()
finally:
    timer.cancel()

print(f"\nDone. Detected {len(event_log)} events.")
if event_log:
    channels_with_events = sorted({e.channel_id for e in event_log})
    print(f"  Active channels: {channels_with_events}")
    durations = [e.duration for e in event_log]
    print(f"  Event durations: min={min(durations):.3f}s, "
          f"max={max(durations):.3f}s, mean={sum(durations)/len(durations):.3f}s")

In [ ]:
# Save results
events_path = writer.save_events()
print(f"Events saved to {events_path}")